# E14: Supplementary Back-Test (Seasonality Coverage)

**Spec sections**: 5.3 (back-test design), 8.4 (report the 8.2 discrimination and rank-ordering metrics), 8.2, 8.3 (bootstrap settings).  
**Inputs**: locked `model_registry.json` (all hashes re-verified; features, params, final rounds), `folds/backtest.parquet` (E3), DEV GOOD+BAD via `common.load_window` (frozen E3 population), `config/feature_dict.csv`.  
**Outputs**: `reports/05_backtest.xlsx`, `reports/run_summaries/s05_backtest.json`, MLflow run. The registry, model files and feature log are **not** modified.

1. Split DEV GOOD+BAD by `folds/backtest.parquet`: train 2022-07..2024-06, test 2024-07..2024-12 (identical to T-fold T3).
2. With the LOCKED feature lists and hyperparameters, refit on the train part: A = BinningProcess (refitted) -> WoE -> unpenalised LR; B = EBM + locked monotonisation; C = monotone LightGBM with rounds = round(final DEV rounds x n_train / n_DEV), from the actual counts.
3. On the test part: AUC, Gini, KS with stratified bootstrap 95% CI, score-band Bad Rates (train-part deciles / quintiles as fixed cut-points), top 10% / 20% Bad capture.

> **Supporting evidence only; the test period was used in feature selection and tuning; not part of the 8.1 decision rules.**

In [0]:
%pip install optbinning "numpy<2" "pyarrow==14.0.1" "statsmodels>=0.14.4" lightgbm interpret-core "scipy>=1.12,<1.14" openpyxl --quiet
dbutils.library.restartPython()

In [0]:
import sys, os, json, math, time, pickle, shutil, tempfile, hashlib, importlib, copy, warnings
from datetime import datetime
sys.path.insert(0, os.getcwd())          # notebook lives in src/
import common
importlib.reload(common)
PROJECT_ROOT = common.find_project_root(os.getcwd())

import numpy as np
import pandas as pd
import sklearn
import lightgbm as lgb
import interpret, optbinning, statsmodels
from interpret.glassbox import ExplainableBoostingClassifier
from optbinning import BinningProcess
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Missing values detected")   # EBM notice; NaN handled natively
warnings.filterwarnings("ignore", message="X does not have valid feature names")

# Config path from widget "config_path" (default config/config.yaml); variant isolation checked
cfg, CONFIG_PATH = common.load_run_config(PROJECT_ROOT, dbutils)
SEED = cfg["random_seed"]
np.random.seed(SEED)
N_THREADS = cfg["runtime"]["lgb_num_threads"]

ARTIFACTS = common.artifacts_path(cfg)
REPORTS   = os.path.join(ARTIFACTS, "reports")
RUN_SUMS  = os.path.join(REPORTS, "run_summaries")
FOLDS_DIR = common.folds_dir(cfg)
REGISTRY_PATH = os.path.join(ARTIFACTS, "model_registry.json")
BT_PATH   = os.path.join(FOLDS_DIR, "backtest.parquet")
XLSX_PATH = os.path.join(REPORTS, "05_backtest.xlsx")
RUN_SUMMARY_PATH = os.path.join(RUN_SUMS, "s05_backtest.json")
for d in (REPORTS, RUN_SUMS):
    os.makedirs(d, exist_ok=True)

cat, sch = cfg["databricks"]["catalog"], cfg["databricks"]["schema"]
MASTER_TBL = common.table_name(cfg, "master")
id_col, date_col, status_col = cfg["columns"]["id"], cfg["columns"]["date"], cfg["columns"]["status"]
LABELLED = cfg["labelled_statuses"]
BT_CFG = cfg["backtest"]                      # train / test date ranges [start, end)
EXP = cfg["expected_counts"]                  # DEV, DEV_EARLY (= back-test train), DEV_LATE (= back-test test)
MODELS = ["A", "B", "C"]

GIT_COMMIT = common.git_commit(PROJECT_ROOT)
CONFIG_HASH = common.config_hash(cfg)
VERSIONS = {"sklearn": sklearn.__version__, "lightgbm": lgb.__version__, "interpret": interpret.__version__,
            "optbinning": optbinning.__version__, "statsmodels": statsmodels.__version__,
            "numpy": np.__version__, "pandas": pd.__version__}

NOTE = ("Supporting evidence only; the test period was used in feature selection and tuning; "
        "not part of the 8.1 decision rules.")

def rnd(x):
    """Round half up (avoid Python banker's rounding), as in E11."""
    return int(math.floor(x + 0.5))

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def read_json(p):
    with open(p) as fh:
        return json.load(fh)

def _json_default(o):
    if hasattr(o, "item"):
        return o.item()
    if isinstance(o, (set, tuple)):
        return list(o)
    return str(o)

def canonical(obj):
    return json.loads(json.dumps(obj, sort_keys=True, default=_json_default))

_TMP = tempfile.mkdtemp(prefix="e14_")
def to_volume(dst, writer):
    """Write to a local temp file, then copy to the UC volume (FUSE does not allow chmod/utime)."""
    tmp = os.path.join(_TMP, os.path.basename(dst))
    writer(tmp)
    shutil.copyfile(tmp, dst)

_t0 = time.time()
print(f"Project root: {PROJECT_ROOT}")
print(f"Config path : {CONFIG_PATH}")
print(f"Git commit  : {GIT_COMMIT[:12]} | Config hash: {CONFIG_HASH[:12]}")
print(f"Folds root  : {FOLDS_DIR}{'  (shared read-only)' if common.folds_shared(cfg) else ''}")
print(f"Back-test   : train {BT_CFG['train']} | test {BT_CFG['test']} | expected DEV_EARLY {EXP['DEV_EARLY']} DEV_LATE {EXP['DEV_LATE']}")

In [0]:
# ── Everything used below comes from the LOCKED registry (spec 5.3: feature list and hyperparameters locked in Step 4) ──
REG = read_json(REGISTRY_PATH)
assert REG.get("locked") is True, "model_registry.json is not locked: run E12 (s04_lock) first"
HASH_CHECK = {k: sha256_file(v["path"]) == v["sha256"] for k, v in REG["files"].items()}
bad = [k for k, ok in HASH_CHECK.items() if not ok]
assert not bad, f"ABORT: {len(bad)} locked file(s) changed since the lock: {bad}"
REG_SNAPSHOT_SHA = hashlib.sha256(json.dumps(canonical(REG), sort_keys=True).encode()).hexdigest()
FILES = {k: v["path"] for k, v in REG["files"].items()}

A_FEATS = REG["features"]["A"]
POOL    = REG["features"]["BC_pool"]
BC_COLS = REG["features"]["BC_columns"]
PA = REG["params"]["A"]
A_FIT_PARAMS = PA["binning_fit_params"]                # L1 binning parameters + special codes per feature
A_CAT = PA["categorical_features"] or []
WOE_KW = PA["woe_transform"]                           # WoE for missing / special rows (as in E11)
B_PARAMS = dict(REG["params"]["B"]["params"])
B_MONO = REG["params"]["B"]["monotonized"]            # {column: increasing}
C_PARAMS = {**REG["params"]["C"]["params"], "monotone_constraints": REG["params"]["C"]["monotone_constraints"]}
C_FINAL_ROUNDS = int(REG["lightgbm_rounds"]["final_rounds"])
CP = read_json(FILES["params/chosen_params.json"])     # hash-verified above; source of the LightGBM categorical list
CAT_COLS = CP["C"]["categorical_feature"]

# Locked feature-list files agree with the registry record
assert [r["feature"] for r in read_json(FILES["feature_lists/model_a.json"])["features"]] == A_FEATS, "model_a.json != registry"
assert [r["feature"] for r in read_json(FILES["feature_lists/l4_pool.json"])] == POOL, "l4_pool.json != registry"
assert set(A_FIT_PARAMS) == set(A_FEATS), "binning_fit_params features != Model A features"
assert B_PARAMS["interactions"] == 0, "Model B must have interactions = 0"
assert set(B_MONO) <= set(BC_COLS), "monotonised EBM terms not in the B/C columns"
assert len(C_PARAMS["monotone_constraints"]) == len(BC_COLS), "monotone vector length != B/C columns"
assert C_PARAMS["num_threads"] == N_THREADS and C_PARAMS["deterministic"] and C_PARAMS["seed"] == SEED

fdict = pd.read_csv(os.path.join(PROJECT_ROOT, "config", "feature_dict.csv"))
print(f"Registry v{REG['version']} locked at {REG['locked_at']}; {len(HASH_CHECK)} file hashes re-verified")
print(f"A: {len(A_FEATS)} features (categorical {A_CAT}) | estimator {PA['estimator']} | WoE kw {WOE_KW}")
print(f"B/C: pool {len(POOL)} -> {len(BC_COLS)} raw columns | B monotonised {len(B_MONO)} | C categorical {CAT_COLS}")
print(f"C final DEV rounds (locked): {C_FINAL_ROUNDS}")

In [0]:
# ── DEV only, through common.load_window (OOT never read) ──
master_dev = spark.table(MASTER_TBL).filter("window = 'DEV'").toPandas()
dev = common.load_window(master_dev, "DEV", cfg)
del master_dev
oot_s, oot_e = cfg["windows"]["OOT"]
_d = pd.to_datetime(dev[date_col])
N_OOT_ROWS_READ = int(((_d >= oot_s) & (_d < oot_e)).sum())
assert N_OOT_ROWS_READ == 0 and set(dev["window"].unique()) == {"DEV"}, "non-DEV / OOT-dated rows loaded"

# DEV GOOD+BAD = the frozen E3 population (as in E10-E13)
frozen_ids = set(pd.read_parquet(FILES["folds/r_folds.parquet"], columns=["app_id"])["app_id"].astype(str))
dev_gb = dev[dev[status_col].isin(LABELLED) & dev[id_col].astype(str).isin(frozen_ids)].reset_index(drop=True)
common.assert_labelled(dev_gb, cfg)
dev_gb["app_id"] = dev_gb[id_col].astype(str)
assert dev_gb["app_id"].is_unique, "app_id not unique in DEV GOOD+BAD"
N_DEV = len(dev_gb)
y_dev = (dev_gb[status_col] == "BAD").astype(int).to_numpy()
assert N_DEV == EXP["DEV"]["records"] and int(y_dev.sum()) == EXP["DEV"]["bads"], "DEV counts differ from expected_counts"

# ── folds/backtest.parquet (E3): part train / valid ──
bt = pd.read_parquet(BT_PATH)
bt["app_id"] = bt["app_id"].astype(str)
BT_SHA = sha256_file(BT_PATH)
assert set(bt["part"]) == {"train", "valid"} and bt["app_id"].is_unique, "backtest.parquet: unexpected parts / duplicate ids"
assert set(bt["app_id"]) <= set(dev_gb["app_id"]), "backtest.parquet ids outside DEV GOOD+BAD"
pos = pd.Series(np.arange(N_DEV), index=dev_gb["app_id"])
TR = np.sort(pos.loc[bt.loc[bt["part"] == "train", "app_id"]].to_numpy())
TE = np.sort(pos.loc[bt.loc[bt["part"] == "valid", "app_id"]].to_numpy())
assert len(np.intersect1d(TR, TE)) == 0, "train / test overlap"

# Dates: every train row in the train range, every test row in the test range, and no DEV GOOD+BAD row missing
d = pd.to_datetime(dev_gb[date_col]).to_numpy()
in_tr = (d >= np.datetime64(BT_CFG["train"][0])) & (d < np.datetime64(BT_CFG["train"][1]))
in_te = (d >= np.datetime64(BT_CFG["test"][0])) & (d < np.datetime64(BT_CFG["test"][1]))
DATES_OK = bool(in_tr[TR].all() and in_te[TE].all() and set(np.where(in_tr)[0]) == set(TR) and set(np.where(in_te)[0]) == set(TE))

# Spec 5.3: the split is identical to time fold T3
tf = pd.read_parquet(FILES["folds/t_folds.parquet"])
tf["app_id"] = tf["app_id"].astype(str)
t3 = tf[tf["fold"] == "T3"]
SAME_AS_T3 = bool(set(t3.loc[t3.part == "train", "app_id"]) == set(dev_gb["app_id"].iloc[TR])
                  and set(t3.loc[t3.part == "valid", "app_id"]) == set(dev_gb["app_id"].iloc[TE]))

train_df = dev_gb.iloc[TR].reset_index(drop=True)
test_df  = dev_gb.iloc[TE].reset_index(drop=True)
y_tr, y_te = y_dev[TR], y_dev[TE]
COUNTS = pd.DataFrame([
    {"part": "train", "range": f"{BT_CFG['train'][0]} .. {BT_CFG['train'][1]} (excl.)", "records": len(TR), "bads": int(y_tr.sum()),
     "bad_rate": y_tr.mean(), "expected_records": EXP["DEV_EARLY"]["records"], "expected_bads": EXP["DEV_EARLY"]["bads"]},
    {"part": "test", "range": f"{BT_CFG['test'][0]} .. {BT_CFG['test'][1]} (excl.)", "records": len(TE), "bads": int(y_te.sum()),
     "bad_rate": y_te.mean(), "expected_records": EXP["DEV_LATE"]["records"], "expected_bads": EXP["DEV_LATE"]["bads"]},
    {"part": "DEV (all)", "range": f"{cfg['windows']['DEV'][0]} .. {cfg['windows']['DEV'][1]} (excl.)", "records": N_DEV,
     "bads": int(y_dev.sum()), "bad_rate": y_dev.mean(), "expected_records": EXP["DEV"]["records"], "expected_bads": EXP["DEV"]["bads"]},
])
print(f"Dates consistent with config.backtest: {DATES_OK} | identical to T3: {SAME_AS_T3} | backtest.parquet sha256 {BT_SHA[:12]}")
display(COUNTS)

In [0]:
# ── Model A: BinningProcess REFITTED on train (locked L1 parameters + special codes) -> WoE -> unpenalised LR ──
def x_a(df):
    X = df[A_FEATS].copy()
    for f in A_FEATS:
        if f not in A_CAT:
            X[f] = pd.to_numeric(X[f], errors="coerce")
    return X

t = time.time()
A_BP_BT = BinningProcess(variable_names=A_FEATS, categorical_variables=A_CAT or None,
                         binning_fit_params=copy.deepcopy(A_FIT_PARAMS)).fit(x_a(train_df), y_tr)
woe = lambda df: np.asarray(A_BP_BT.transform(x_a(df), metric="woe", **WOE_KW), dtype=float)
A_LR_BT = common.lr_unpenalised(max_iter=2000).fit(woe(train_df), y_tr)       # same estimator factory as E11
A_BIN_TOTALS = {f: int(A_BP_BT.get_binned_variable(f).binning_table.build().loc["Totals", "Count"]) for f in A_FEATS}
print(f"A fitted in {time.time() - t:.1f}s | binning Totals = n_train for all features: "
      f"{all(v == len(TR) for v in A_BIN_TOTALS.values())}")

# ── Raw matrix for B and C (special codes -> NaN + indicator), built from each part's own rows ──
XR_TR = common.prepare_raw(train_df, POOL, fdict)
XR_TE = common.prepare_raw(test_df, POOL, fdict)
assert list(XR_TR.columns) == BC_COLS and list(XR_TE.columns) == BC_COLS, "prepare_raw columns differ from the locked B/C columns"

# ── Model B: EBM with the locked params, then the locked monotonisation ──
t = time.time()
B_EBM_BT = ExplainableBoostingClassifier(**B_PARAMS).fit(XR_TR, y_tr)
for f, inc in B_MONO.items():
    B_EBM_BT.monotonize(f, increasing=bool(inc))
assert all(len(tf_) == 1 for tf_ in B_EBM_BT.term_features_), "EBM contains interaction terms"
print(f"B fitted in {time.time() - t:.1f}s | {len(B_MONO)} terms monotonised")

# ── Model C: locked params + monotone vector; rounds scaled by sample size from the ACTUAL counts (spec 5.3) ──
N_TR = len(TR)
C_ROUNDS_BT = rnd(C_FINAL_ROUNDS * N_TR / N_DEV)
C_ROUNDS_SPEC_LITERAL = rnd(C_FINAL_ROUNDS * 7612 / 10526)    # spec 1.2 counts, for reference only
t = time.time()
ds = lgb.Dataset(XR_TR, label=y_tr, categorical_feature=CAT_COLS, free_raw_data=False,
                 params={"feature_pre_filter": False, "verbose": -1})
C_BST_BT = lgb.train(C_PARAMS, ds, num_boost_round=C_ROUNDS_BT)
print(f"C fitted in {time.time() - t:.1f}s | rounds = round({C_FINAL_ROUNDS} x {N_TR} / {N_DEV}) = {C_ROUNDS_BT} "
      f"(spec-count version {C_ROUNDS_SPEC_LITERAL}, not used) | trees {C_BST_BT.num_trees()}")

def predict(df, XR):
    return {"A": A_LR_BT.predict_proba(woe(df))[:, 1], "B": B_EBM_BT.predict_proba(XR)[:, 1], "C": C_BST_BT.predict(XR)}
P_TR = predict(train_df, XR_TR)          # in-sample; used only for the fixed band cut-points
P_TE = predict(test_df, XR_TE)
for part, P in (("train", P_TR), ("test", P_TE)):
    for m in MODELS:
        assert np.isfinite(P[m]).all() and ((P[m] > 0) & (P[m] < 1)).all(), f"{part} {m}: PD outside (0, 1)"
print({m: round(roc_auc_score(y_te, P_TE[m]), 4) for m in MODELS}, "<- test AUC (point)")

In [0]:
# Bootstrap settings from the registry (pre-registered, spec 8.3): stratified, 2,000 resamples, seed 42
cfg_boot = copy.deepcopy(cfg)
cfg_boot["selection"]["bootstrap"] = REG["pre_registered"]["bootstrap"]

def capture(y, p, share):
    k = int(np.ceil(share * len(p)))
    top = np.argsort(-p, kind="mergesort")[:k]
    return float(y[top].sum() / y.sum())

def edges(p, q):
    return np.unique(np.quantile(p, np.linspace(0, 1, q + 1)[1:-1]))

def band(p, e):
    return np.searchsorted(e, p, side="right") + 1          # 1 = lowest PD (best) ... q = highest PD

DEV_OOF = REG["dev_oof_metrics"]
rows = []
for m in MODELS:
    r = {"model": m, **common.bootstrap_auc_gini_ks(y_te, P_TE[m], cfg_boot),
         "top10_capture": capture(y_te, P_TE[m], 0.10), "top20_capture": capture(y_te, P_TE[m], 0.20),
         "train_auc_in_sample": roc_auc_score(y_tr, P_TR[m]),
         "dev_oof_auc_locked": DEV_OOF[m]["auc"], "dev_oof_gini_locked": DEV_OOF[m]["gini"], "dev_oof_ks_locked": DEV_OOF[m]["ks"]}
    rows.append(r)
METRICS = pd.DataFrame(rows)

# Score bands: train-part deciles / quintiles of each back-test model's PD as fixed cut-points, applied to the test part
# (the back-test analogue of "DEV deciles applied to OOT", spec 8.2). Ranking only, so raw PD is used (no calibration).
EDGES = {m: {"decile": edges(P_TR[m], 10), "quintile": edges(P_TR[m], 5)} for m in MODELS}
band_tabs, mono_rows = {}, []
for kind in ("decile", "quintile"):
    parts = []
    for m in MODELS:
        e = EDGES[m][kind]
        t = pd.DataFrame({"band": band(P_TE[m], e), "y": y_te, "pd": P_TE[m]})
        g = (t.groupby("band").agg(n=("y", "size"), bads=("y", "sum"), mean_pd_raw=("pd", "mean"),
                                   min_pd=("pd", "min"), max_pd=("pd", "max"))
               .reindex(range(1, len(e) + 2)).reset_index())
        g["bad_rate"] = g["bads"] / g["n"]
        g["share_test"] = g["n"] / len(t)
        g["share_train"] = pd.Series(band(P_TR[m], e)).value_counts(normalize=True).reindex(g["band"]).to_numpy()
        g.insert(0, "model", m)
        parts.append(g)
        br = g["bad_rate"].dropna().to_numpy()
        n_rev = int((np.diff(br) < 0).sum())
        mono_rows.append({"model": m, "kind": kind, "n_bands": len(e) + 1, "empty_bands": int(g["n"].isna().sum()),
                          "monotonic": n_rev == 0, "n_reversals": n_rev,
                          "top_vs_bottom_bad_rate_ratio": float(br[-1] / br[0]) if br[0] > 0 else np.inf})
    band_tabs[kind] = pd.concat(parts, ignore_index=True)
BAND_MONO = pd.DataFrame(mono_rows)

print(METRICS[["model", "n", "n_bad", "auc", "auc_lo", "auc_hi", "gini", "gini_lo", "gini_hi", "ks", "ks_lo", "ks_hi",
               "top10_capture", "top20_capture", "dev_oof_gini_locked"]].round(4).to_string(index=False))
print(BAND_MONO.round(3).to_string(index=False))
display(band_tabs["quintile"].round(4))

In [0]:
# The back-test split is T3, which E11 already fitted with the same features / params (A and B identical procedure;
# C used stage-3 best_iter scaling instead of the 5.3 rule). Agreement is a cross-check, not an acceptance criterion.
E11_XLSX = os.path.join(REPORTS, "04_training.xlsx")
T3_XCHECK = pd.DataFrame()
try:
    tfm = pd.read_excel(E11_XLSX, sheet_name="tfold_metrics")
    print("E11 tfold_metrics columns:", list(tfm.columns))
    fold_c = next(c for c in tfm.columns if c.lower() in ("fold", "tfold", "t_fold", "name"))
    t3m = tfm[tfm[fold_c].astype(str) == "T3"].set_index("model")
    T3_XCHECK = pd.DataFrame([{"model": m, "e11_t3_auc": float(t3m.loc[m, "auc"]),
                               "e14_test_auc": float(METRICS.set_index("model").loc[m, "auc"]),
                               "e11_t3_rounds": t3m.loc[m, "rounds"] if "rounds" in t3m.columns else None} for m in MODELS])
    T3_XCHECK["diff"] = T3_XCHECK["e14_test_auc"] - T3_XCHECK["e11_t3_auc"]
    T3_XCHECK["note"] = ["same procedure", "same procedure", "rounds rule differs (E11: best_iter scaling)"]
    display(T3_XCHECK.round(5))
except Exception as ex:
    print(f"E11 T3 cross-check not available: {type(ex).__name__}: {ex}")

In [0]:
RUN_TIME = datetime.now().isoformat(timespec="seconds")
META = pd.DataFrame([{"module": "E14_backtest", "run_time": RUN_TIME, "git_commit": GIT_COMMIT, "config_sha256": CONFIG_HASH,
                      "registry_version": REG["version"], "registry_locked_at": REG["locked_at"],
                      "backtest_parquet_sha256": BT_SHA, "note": NOTE, **VERSIONS}])
NOTES = pd.DataFrame({"note": [
    NOTE,
    "Spec 5.3 / 8.4: the test period (2024-07..2024-12) belongs to DEV and cannot replace OOT. The model document MUST say so.",
    f"Train {BT_CFG['train'][0]}..{BT_CFG['train'][1]} (excl.) = {len(TR):,} / {int(y_tr.sum())} Bads; "
    f"test {BT_CFG['test'][0]}..{BT_CFG['test'][1]} (excl.) = {len(TE):,} / {int(y_te.sum())} Bads (DEV GOOD+BAD, frozen E3 population). "
    f"Identical to T-fold T3: {SAME_AS_T3}.",
    "Features and hyperparameters: the locked registry values (A features, B/C pool and columns, A binning params, "
    "B EBM params and monotonisation, C LightGBM params and monotone vector). Binning and models are refitted on the train part only.",
    f"LightGBM rounds = round(final DEV rounds {C_FINAL_ROUNDS} x n_train {N_TR} / n_DEV {N_DEV}) = {C_ROUNDS_BT} (actual counts; "
    f"the spec 1.2 counts 7,612 / 10,526 would give {C_ROUNDS_SPEC_LITERAL}).",
    "AUC / Gini / KS on raw PD with stratified bootstrap 95% CI (registry bootstrap settings). Score bands: train-part deciles / "
    "quintiles of each back-test model's PD as fixed cut-points applied to the test part; band 1 = lowest PD.",
    "No calibration is applied (rank-ordering metrics only); the registry, model files and feature log are not modified.",
]})
ROUNDS = pd.DataFrame([{"final_dev_rounds_locked": C_FINAL_ROUNDS, "n_train": N_TR, "n_dev": N_DEV,
                        "backtest_rounds": C_ROUNDS_BT, "trees_in_booster": C_BST_BT.num_trees(),
                        "spec_literal_rounds_7612_10526": C_ROUNDS_SPEC_LITERAL}])
A_COEF = pd.DataFrame({"feature": A_FEATS, "coef_backtest": A_LR_BT.coef_.ravel(),
                       "n_bins_backtest": [len(A_BP_BT.get_binned_variable(f).binning_table.build()) - 1 for f in A_FEATS]})

def _write_xlsx(p):
    with pd.ExcelWriter(p, engine="openpyxl") as w:
        META.to_excel(w, sheet_name="meta", index=False)
        NOTES.to_excel(w, sheet_name="note", index=False)
        COUNTS.to_excel(w, sheet_name="counts", index=False)
        METRICS.to_excel(w, sheet_name="discrimination", index=False)
        band_tabs["decile"].to_excel(w, sheet_name="bands_decile", index=False)
        band_tabs["quintile"].to_excel(w, sheet_name="bands_quintile", index=False)
        BAND_MONO.to_excel(w, sheet_name="band_monotonicity", index=False)
        ROUNDS.to_excel(w, sheet_name="lgb_rounds", index=False)
        A_COEF.to_excel(w, sheet_name="A_backtest_coefficients", index=False)
        if len(T3_XCHECK):
            T3_XCHECK.to_excel(w, sheet_name="e11_t3_crosscheck", index=False)
to_volume(XLSX_PATH, _write_xlsx)
print(f"Saved {XLSX_PATH} ({len(pd.ExcelFile(XLSX_PATH).sheet_names)} sheets)")

In [0]:
import mlflow
mlflow.set_experiment(cfg["databricks"]["mlflow_experiment"])
met = METRICS.set_index("model")
with mlflow.start_run(run_name="E14_backtest") as run:
    mlflow.set_tags({"module": "E14_backtest", "git_commit": GIT_COMMIT, "registry_version": REG["version"],
                     "status": "supporting_evidence_only"})
    mlflow.log_params({"config_hash": CONFIG_HASH, "seed": SEED, "n_train": N_TR, "n_test": len(TE), "n_dev": N_DEV,
                       "c_final_rounds": C_FINAL_ROUNDS, "c_backtest_rounds": C_ROUNDS_BT,
                       "backtest_parquet_sha256": BT_SHA[:16]})
    mlflow.log_metrics({f"{m}_test_{k}": float(met.loc[m, k]) for m in MODELS
                        for k in ("auc", "auc_lo", "auc_hi", "gini", "gini_lo", "gini_hi", "ks", "ks_lo", "ks_hi",
                                  "top10_capture", "top20_capture")})
    mlflow.log_artifact(XLSX_PATH)
    MLFLOW_RUN_ID = run.info.run_id
print(f"MLflow run: {MLFLOW_RUN_ID}")

In [0]:
# ── Acceptance (E14): re-read outputs and check every criterion ──
print("=" * 60); print("ACCEPTANCE CRITERIA (E14 — supplementary back-test)"); print("=" * 60)
REG2 = read_json(REGISTRY_PATH)
reg_unchanged = hashlib.sha256(json.dumps(canonical(REG2), sort_keys=True).encode()).hexdigest() == REG_SNAPSHOT_SHA
same = lambda a, b: canonical(a) == canonical(b)

# Features actually used by the fitted objects vs the registry
feat_ok = (list(A_BP_BT.variable_names) == REG2["features"]["A"] and A_LR_BT.coef_.shape[1] == len(REG2["features"]["A"])
           and list(B_EBM_BT.feature_names_in_) == REG2["features"]["BC_columns"]
           and C_BST_BT.feature_name() == REG2["features"]["BC_columns"])
# Params actually used vs the registry
ebm_used = B_EBM_BT.get_params()
bst_used = C_BST_BT.params
params_ok = (same(A_BP_BT.binning_fit_params, REG2["params"]["A"]["binning_fit_params"])
             and same({k: ebm_used[k] for k in REG2["params"]["B"]["params"]}, REG2["params"]["B"]["params"])
             and same(B_MONO, REG2["params"]["B"]["monotonized"])
             and same({k: bst_used[k] for k in REG2["params"]["C"]["params"]}, REG2["params"]["C"]["params"])
             and same(list(bst_used["monotone_constraints"]), REG2["params"]["C"]["monotone_constraints"]))
rounds_expected = rnd(REG2["lightgbm_rounds"]["final_rounds"] * len(TR) / N_DEV)
xl = pd.ExcelFile(XLSX_PATH)
meta_r = pd.read_excel(XLSX_PATH, sheet_name="meta")
note_r = pd.read_excel(XLSX_PATH, sheet_name="note")
disc_r = pd.read_excel(XLSX_PATH, sheet_name="discrimination")
bands_r = {k: pd.read_excel(XLSX_PATH, sheet_name=f"bands_{k}") for k in ("decile", "quintile")}

# Independent AUC check: Mann-Whitney rank formula vs the bootstrap helper's point estimate
from scipy.stats import rankdata
def auc_rank(y, p):
    rk = rankdata(p); n1 = y.sum(); n0 = len(y) - n1
    return (rk[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
auc_diff = max(abs(auc_rank(y_te, P_TE[m]) - METRICS.set_index("model").loc[m, "auc"]) for m in MODELS)

checks = [
    ("Registry locked, all locked file hashes match, registry content unchanged by E14",
     REG2.get("locked") is True and all(sha256_file(v["path"]) == v["sha256"] for v in REG2["files"].values()) and reg_unchanged),
    (f"Features equal the registry (A {len(A_FEATS)}; B/C {len(BC_COLS)} columns) in every fitted model", feat_ok),
    ("Params equal the registry (A binning_fit_params; B EBM params + monotonisation; C LightGBM params + monotone vector)", params_ok),
    (f"Train counts = DEV_EARLY {EXP['DEV_EARLY']} (actual {len(TR)} / {int(y_tr.sum())})",
     len(TR) == EXP["DEV_EARLY"]["records"] and int(y_tr.sum()) == EXP["DEV_EARLY"]["bads"]),
    (f"Test counts = DEV_LATE {EXP['DEV_LATE']} (actual {len(TE)} / {int(y_te.sum())})",
     len(TE) == EXP["DEV_LATE"]["records"] and int(y_te.sum()) == EXP["DEV_LATE"]["bads"]),
    ("Train/test disjoint, dates match config.backtest, split identical to T3, only DEV GOOD+BAD read",
     DATES_OK and SAME_AS_T3 and N_OOT_ROWS_READ == 0 and set(dev_gb[status_col]) <= set(LABELLED)),
    (f"Model A binning refitted on train rows only (Totals = {len(TR)} for all {len(A_FEATS)} features)",
     all(v == len(TR) for v in A_BIN_TOTALS.values())),
    (f"LightGBM rounds = round({REG2['lightgbm_rounds']['final_rounds']} x {len(TR)} / {N_DEV}) = {rounds_expected}; booster has "
     f"{C_BST_BT.num_trees()} trees", C_ROUNDS_BT == rounds_expected == C_BST_BT.num_trees()),
    ("AUC, Gini, KS with 95% CI and top 10/20% capture for A, B, C in the report",
     set(disc_r["model"]) == set(MODELS) and disc_r[["auc_lo", "auc_hi", "gini_lo", "gini_hi", "ks_lo", "ks_hi",
                                                     "top10_capture", "top20_capture"]].notna().all().all()
     and bool((disc_r["auc_lo"] <= disc_r["auc"]).all() and (disc_r["auc"] <= disc_r["auc_hi"]).all())),
    (f"AUC matches the rank (Mann-Whitney) formula (max diff {auc_diff:.2e})", auc_diff < 1e-9),
    ("Decile and quintile Bad Rate tables for every model; test counts add up per model",
     all(set(bands_r[k]["model"]) == set(MODELS) and all(int(g["n"].sum()) == len(TE) and int(g["bads"].sum()) == int(y_te.sum())
                                                         for _, g in bands_r[k].groupby("model")) for k in bands_r)),
    ("Report: first sheet 'meta' (module, run time, git commit, config sha256) and the supporting-evidence note",
     xl.sheet_names[0] == "meta" and {"module", "run_time", "git_commit", "config_sha256"} <= set(meta_r.columns)
     and meta_r.loc[0, "note"] == NOTE and note_r.loc[0, "note"] == NOTE),
]
for name, ok in checks:
    print(f"  {'✅ PASS' if ok else '❌ FAIL'} : {name}")
ALL_PASS = all(ok for _, ok in checks)

RUN_SUMMARY = {
    "module": "E14_backtest", "run_time": RUN_TIME, "duration_s": round(time.time() - _t0, 1), "git_commit": GIT_COMMIT,
    "config_sha256": CONFIG_HASH, "registry_version": REG2["version"], "mlflow_run_id": MLFLOW_RUN_ID,
    "row_counts": {"dev_gb": N_DEV, "train": len(TR), "train_bads": int(y_tr.sum()), "test": len(TE), "test_bads": int(y_te.sum())},
    "lgb_rounds": {"final": C_FINAL_ROUNDS, "backtest": C_ROUNDS_BT},
    "metrics": {m: {k: round(float(METRICS.set_index("model").loc[m, k]), 6)
                    for k in ("auc", "auc_lo", "auc_hi", "gini", "gini_lo", "gini_hi", "ks", "ks_lo", "ks_hi", "top10_capture", "top20_capture")}
                for m in MODELS},
    "band_monotonic": {f"{r.model}_{r.kind}": bool(r.monotonic) for r in BAND_MONO.itertuples()},
    "note": NOTE,
    "acceptance": {name: bool(ok) for name, ok in checks}, "all_pass": ALL_PASS,
}
to_volume(RUN_SUMMARY_PATH, lambda p: json.dump(RUN_SUMMARY, open(p, "w"), indent=2, default=_json_default))
print(f"\nRun summary: {RUN_SUMMARY_PATH}")

for name, ok in checks:
    assert ok, f"E14 acceptance failed: {name}"
print(f"\nE14 acceptance: {len(checks)}/{len(checks)} PASS")